In [1]:
import pandas as pd

In [2]:
# Download required NLTK data
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\esraa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
def load_data():
    df = pd.read_csv("Medical Diagnosis Expert System.csv")
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    df = df.drop_duplicates()
    df = df.dropna()

    df["disease"] = df["disease"].str.strip().str.lower().str.replace(" ", "_")
    df["symptoms"] = df["symptoms"].apply(
        lambda x: [s.strip().lower().replace(" ", "_") for s in x.split(",")]
    )
    df["precautions"] = df["precautions"].apply(
        lambda x: [p.strip().lower() for p in x.split(",")]
    )

    df = df.reset_index(drop=True)
    df["pattern_id"] = df.index

    patterns = []
    for _, row in df.iterrows():
        patterns.append({
            "pattern_id": row["pattern_id"],
            "disease": row["disease"],
            "symptoms": row["symptoms"],
            "precautions": row["precautions"]
        })

    all_symptoms = sorted(set(
        symptom
        for pattern in patterns
        for symptom in pattern["symptoms"]
    ))

    matrix_data = []
    for pattern in patterns:
        row_vector = [1 if symptom in pattern["symptoms"] else 0
                      for symptom in all_symptoms]
        matrix_data.append(row_vector)

    pattern_matrix = pd.DataFrame(
        matrix_data,
        columns=all_symptoms,
        index=df["pattern_id"]
    )

    disease_index = {}
    for pattern in patterns:
        disease = pattern["disease"]
        pid = pattern["pattern_id"]
        if disease not in disease_index:
            disease_index[disease] = []
        disease_index[disease].append(pid)

    return patterns, pattern_matrix, all_symptoms, disease_index

In [4]:
patterns, pattern_matrix, all_symptoms, disease_index = load_data()

print("Patterns count:", len(patterns))
print("Unique symptoms:", len(all_symptoms))
print("Matrix shape:", pattern_matrix)
print("Unique diseases:", len(disease_index))

# Pick first pattern and verify matrix matches
p = patterns[0]
print("\nPattern 0 disease:", p["disease"])
print("Pattern 0 symptoms:", p["symptoms"])
print("Matrix row 0 (only 1s):", 
      pattern_matrix.loc[0][pattern_matrix.loc[0] == 1].index.tolist())
# These two lists must match exactly

Patterns count: 288
Unique symptoms: 128
Matrix shape:             abdominal_pain  abnormal_menstruation  acidity  \
pattern_id                                                   
0                        0                      0        0   
1                        0                      0        0   
2                        0                      0        0   
3                        0                      0        0   
4                        0                      0        0   
...                    ...                    ...      ...   
283                      0                      0        0   
284                      0                      0        0   
285                      0                      0        0   
286                      0                      0        0   
287                      0                      0        0   

            acute_liver_failure  altered_sensorium  anxiety  back_pain  \
pattern_id                                                      

In [39]:
all_symptoms

['abdominal_pain',
 'abnormal_menstruation',
 'acidity',
 'acute_liver_failure',
 'altered_sensorium',
 'anxiety',
 'back_pain',
 'belly_pain',
 'blackheads',
 'bladder_discomfort',
 'blister',
 'blood_in_sputum',
 'bloody_stool',
 'blurred_and_distorted_vision',
 'breathlessness',
 'brittle_nails',
 'bruising',
 'burning_micturition',
 'chest_pain',
 'chills',
 'cold_hands_and_feets',
 'coma',
 'congestion',
 'constipation',
 'continuous_feel_of_urine',
 'continuous_sneezing',
 'cough',
 'cramps',
 'dark_urine',
 'dehydration',
 'depression',
 'diarrhoea',
 'dischromic__patches',
 'distention_of_abdomen',
 'dizziness',
 'drying_and_tingling_lips',
 'enlarged_thyroid',
 'excessive_hunger',
 'extra_marital_contacts',
 'family_history',
 'fast_heart_rate',
 'fatigue',
 'fluid_overload',
 'foul_smell_of_urine',
 'headache',
 'high_fever',
 'hip_joint_pain',
 'history_of_alcohol_consumption',
 'increased_appetite',
 'indigestion',
 'inflammatory_nails',
 'internal_itching',
 'irregular_sug

In [5]:
# Tokenization using NLTK
from functools import reduce
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
tokenizer = RegexpTokenizer(r'\w+')

In [6]:
stemmed_symptom_dict = {}

for symptom in all_symptoms:
    
    words = symptom.replace("_", " ")
    words = tokenizer.tokenize(words)
    words = [w for w in words if w not in stop_words]
    
    stemmed_words = frozenset(ps.stem(w) for w in words)
    stemmed_symptom_dict[stemmed_words] = symptom

In [7]:
sent = "i have abdominal pain and acidity and belly pain what to do i feel like why has had of on one two over in "
tokens = tokenizer.tokenize(sent)

filtered_tokens = [word for word in tokens if word not in stop_words]

print("filtered tokens = ",filtered_tokens)
stemmed_s = []
for w in filtered_tokens:
    w.replace("_"," ")
    w = ps.stem(w)
    stemmed_s.append(w)
print("stemmed tokens = ",stemmed_s)

matched = []
for stemmed_key, original_symptom in stemmed_symptom_dict.items():
    if stemmed_key.issubset(stemmed_s):
        matched.append(original_symptom)
        
print("symptom = ",matched)

if len(matched) == 0:
        print("\nSorry, I could not recognize any symptoms from what you described.")
        print("Please try to describe your symptoms more clearly.")
        print("Example: 'I have itching, skin rash and fever'")

filtered tokens =  ['abdominal', 'pain', 'acidity', 'belly', 'pain', 'feel', 'like', 'one', 'two']
stemmed tokens =  ['abdomin', 'pain', 'acid', 'belli', 'pain', 'feel', 'like', 'one', 'two']
symptom =  ['abdominal_pain', 'acidity', 'belly_pain']


In [8]:
stemmed_symptom_dict

{frozenset({'abdomin', 'pain'}): 'abdominal_pain',
 frozenset({'abnorm', 'menstruat'}): 'abnormal_menstruation',
 frozenset({'acid'}): 'acidity',
 frozenset({'acut', 'failur', 'liver'}): 'acute_liver_failure',
 frozenset({'alter', 'sensorium'}): 'altered_sensorium',
 frozenset({'anxieti'}): 'anxiety',
 frozenset({'back', 'pain'}): 'back_pain',
 frozenset({'belli', 'pain'}): 'belly_pain',
 frozenset({'blackhead'}): 'blackheads',
 frozenset({'bladder', 'discomfort'}): 'bladder_discomfort',
 frozenset({'blister'}): 'blister',
 frozenset({'blood', 'sputum'}): 'blood_in_sputum',
 frozenset({'bloodi', 'stool'}): 'bloody_stool',
 frozenset({'blur', 'distort', 'vision'}): 'blurred_and_distorted_vision',
 frozenset({'breathless'}): 'breathlessness',
 frozenset({'brittl', 'nail'}): 'brittle_nails',
 frozenset({'bruis'}): 'bruising',
 frozenset({'burn', 'micturit'}): 'burning_micturition',
 frozenset({'chest', 'pain'}): 'chest_pain',
 frozenset({'chill'}): 'chills',
 frozenset({'cold', 'feet', 'h

In [9]:
sent = "i have abdominal pain and acidity and belly pain what to do i feel like why has had of on one two over in "
tokens = tokenizer.tokenize(sent)

matched = []
for stemmed_key, original_symptom in stemmed_symptom_dict.items():
    if stemmed_key.issubset(tokens):
        matched.append(original_symptom)
        
print(matched)

[]


In [10]:
all_symptoms_stemmed = []

for symptom in all_symptoms:
    words = symptom.replace("_"," ")
    words = tokenizer.tokenize(words)
    words = [word for word in words if word not in stop_words]
    symptoms = []

    for w in words:
        w = ps.stem(w)
        symptoms.append(w)

    symptoms = "_".join(symptoms)
    all_symptoms_stemmed.append(symptoms)

In [11]:
import data_prep as data
from functools import reduce
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
tokenizer = RegexpTokenizer(r'\w+')

patterns, all_symptoms = data.load_data()

def load_stemmed_symptom_dict():
    
    stemmed_symptom_dict = {}
    for symptom in all_symptoms:

        words = symptom.replace("_", " ")
        words = tokenizer.tokenize(words)
        words = [w for w in words if w not in stop_words]

        stemmed_words = frozenset(ps.stem(w) for w in words)
        stemmed_symptom_dict[stemmed_words] = symptom
    
    return stemmed_symptom_dict

stemmed_symptom_dict = load_stemmed_symptom_dict()

def process_user_input(user_input):
    
    tokens = tokenizer.tokenize(user_input)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    
    stemmed_tokens = []
    for w in filtered_tokens:
        w = w.replace("_"," ")
        w = ps.stem(w)
        stemmed_tokens.append(w)
    
    return stemmed_tokens

def match_symptoms(stemmed_tokens):
    matched_symptoms = []
    for stemmed_key, original_symptom in stemmed_symptom_dict.items():
        if stemmed_key.issubset(stemmed_tokens):
            matched_symptoms.append(original_symptom)
    return matched_symptoms

def no_matched_symptoms(matched_symptoms):
    if len(matched_symptoms) == 0:
        print("\nSorry, I could not recognize any symptoms from what you described.")
        print("Please try to describe your symptoms more clearly.")
        print("Example: 'I have itching, skin rash and fever'")
        
def extract_symptoms(user_input):
    stemmed_tokens = process_user_input(user_input)
    matched_symptoms = match_symptoms(stemmed_tokens)
    no_matched_symptoms(matched_symptoms)
    print(matched_symptoms)
    return matched_symptoms
    
text = input("Describe your symptoms: ")
matched_symptoms=extract_symptoms(text)




['dischromic__patches', 'itching', 'nodal_skin_eruptions', 'skin_rash']


In [14]:
from experta import *

class diseases_symptom(Fact):
    pass
class dieases_matched(Fact):
    pass
class MedicalExpertSystem(KnowledgeEngine):
    def __init__(self,matched_symptoms,knowledge_base):
        super().__init__()
        self.matched_symptoms = matched_symptoms
        self.knowledge_base = knowledge_base
    @DefFacts()
    def _initial_action(self):
        # yield Fact(action="find_disease")
        for symptom in self.matched_symptoms:
            yield diseases_symptom(name=symptom)
    @Rule()
    def diagonise(self):
        for obj in self.knowledge_base:
            #i want to get the intersection with eah disease and if it is not empty, add the matched as facts and missed symptomps also 
            matched = set(obj["symptoms"]).intersection(set(self.matched_symptoms))
            if matched:
                missed = set(obj["symptoms"]).difference(set(self.matched_symptoms))
                self.declare(dieases_matched(id=obj["id"],
                                             name=obj["name"],
                                            matched=list(matched), 
                                            missed=list(missed),
                                            counter=len(matched)
                                            ))

engine = MedicalExpertSystem(matched_symptoms,patterns)
engine.reset()    
engine.run()         
#i want to print   dieases_matched that is knowledge base
results_for_logic_lead = []

for fact in engine.facts.values():
    if isinstance(fact, dieases_matched):
        if fact['counter']/len(fact['matched'] + fact['missed']) >= 0.2:  # Only include diseases with at least 50% symptom match
           results_for_logic_lead.append(dict(fact))

for res in results_for_logic_lead:
    print(res)    
len(results_for_logic_lead)

{'id': 0, 'name': 'fungal_infection', 'matched': frozenlist(['nodal_skin_eruptions', 'skin_rash', 'itching', 'dischromic__patches']), 'missed': frozenlist([]), 'counter': 4, '__factid__': 5}
{'id': 0, 'name': 'fungal_infection', 'matched': frozenlist(['nodal_skin_eruptions', 'skin_rash', 'dischromic__patches']), 'missed': frozenlist([]), 'counter': 3, '__factid__': 6}
{'id': 0, 'name': 'fungal_infection', 'matched': frozenlist(['nodal_skin_eruptions', 'itching', 'dischromic__patches']), 'missed': frozenlist([]), 'counter': 3, '__factid__': 7}
{'id': 0, 'name': 'fungal_infection', 'matched': frozenlist(['skin_rash', 'itching', 'dischromic__patches']), 'missed': frozenlist([]), 'counter': 3, '__factid__': 8}
{'id': 0, 'name': 'fungal_infection', 'matched': frozenlist(['nodal_skin_eruptions', 'skin_rash', 'itching']), 'missed': frozenlist([]), 'counter': 3, '__factid__': 9}
{'id': 0, 'name': 'chicken_pox', 'matched': frozenlist(['skin_rash', 'itching']), 'missed': frozenlist(['lethargy', 

26